# 🎨 ComfyUI HYPHA Studio
Darmowe studio graficzne AI na GPU Google Colab.
- **Flux** — najlepszy model do obrazów
- **Stable Audio** — generowanie dźwięków
- **Wideo AI** — krótkie klipy

⚠️ Wybierz GPU: Runtime → Change runtime type → T4 GPU

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'BRAK — zmień runtime na T4 GPU!'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")

## 1. Instalacja ComfyUI

In [ ]:
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -r requirements.txt -q
!pip install -q aiohttp einops transformers safetensors accelerate

## 2. Pobierz modele
Wybierz co potrzebujesz:

In [ ]:
# Flux Schnell — szybki (4 kroki), darmowy, dobra jakość
import os
os.makedirs("/content/ComfyUI/models/unet", exist_ok=True)
os.makedirs("/content/ComfyUI/models/clip", exist_ok=True)
os.makedirs("/content/ComfyUI/models/vae", exist_ok=True)

# Flux Schnell (fp8 — mieści się w 16GB VRAM)
!wget -q -O /content/ComfyUI/models/unet/flux1-schnell-fp8.safetensors \
  "https://huggingface.co/Comfy-Org/flux1-schnell/resolve/main/flux1-schnell-fp8_e4m3fn.safetensors"

# CLIP encoders
!wget -q -O /content/ComfyUI/models/clip/clip_l.safetensors \
  "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors"
!wget -q -O /content/ComfyUI/models/clip/t5xxl_fp8_e4m3fn.safetensors \
  "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors"

# VAE
!wget -q -O /content/ComfyUI/models/vae/ae.safetensors \
  "https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors"

print("✅ Flux Schnell gotowy!")

## 3. Uruchom ComfyUI
Po uruchomieniu kliknij link z cloudflared tunnel.

In [ ]:
# Pobierz cloudflared do tunelowania
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import subprocess
import threading
import time
import re

# Uruchom cloudflared tunnel w tle
def run_tunnel():
    process = subprocess.Popen(
        ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    for line in process.stderr:
        if 'trycloudflare.com' in line:
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if match:
                print(f"\n🟢 ComfyUI dostępne pod: {match.group()}\n")

tunnel_thread = threading.Thread(target=run_tunnel, daemon=True)
tunnel_thread.start()

time.sleep(3)

# Uruchom ComfyUI
!cd /content/ComfyUI && python main.py --listen 0.0.0.0 --port 8188

## 4. Gotowe prompty dla naszych projektów

### Kolekcja Głuchowskich (tła katalogowe)
```
professional auction catalog background, dark charcoal gray, subtle gradient, 
elegant museum lighting, Christie's Sotheby's style, minimalist, 8k
```

### Opera Prometeus (wizualizacje)
```
ethereal fire figure singing, prometheus myth, golden chains dissolving,
dark theatrical stage, dramatic lighting, opera scenography, cinematic, 8k
```

### HYPHA / Membrana (tekstury)
```
organic membrane texture, bioluminescent, deep ocean, translucent surface,
mycelium network glowing, abstract biological art, dark background, 8k
```

### Katedra (warstwy wizualne)
```
cathedral interior, light beams through stained glass, ethereal atmosphere,
hands pressing through translucent membrane, subsurface scattering, 8k
```

## 5. Stable Audio (bonus)
Generowanie dźwięków do HYPHA.

In [ ]:
# Stable Audio — generowanie ambient/efektów dźwiękowych
!pip install -q stable-audio-tools

from stable_audio_tools import get_pretrained_model
import torch
import torchaudio

# Model (wymaga akceptacji licencji na HuggingFace)
# Odkomentuj po zalogowaniu: huggingface-cli login
# model, model_config = get_pretrained_model("stabilityai/stable-audio-open-1.0")

print("ℹ️ Stable Audio wymaga tokena HuggingFace.")
print("Uruchom: !huggingface-cli login")
print("Zaakceptuj licencję: https://huggingface.co/stabilityai/stable-audio-open-1.0")